# 05 · MCP server: capability discovery and use

The statistics store is exposed through an MCP server (`src/policy_agent/mcp_server/`). This notebook starts it over
stdio through the validating client (`policy_agent.mcp_client`) and walks through what a host discovers and can call.
The raw JSON-RPC lifecycle and the failure cases are in `scripts/mcp_protocol_demo.py`.

In [1]:
from policy_agent import mcp_client as mc

allow = mc.load_allow_list()
print("allow-listed servers:", list(allow["servers"]))
print("expected tools:", allow["servers"]["stats"]["expected_tools"])

allow-listed servers: ['stats']
expected tools: ['query_statistic', 'compute_change']


## 1. Connect: handshake and checks
The client refuses a server that is not allow-listed, speaks an unexpected protocol version, or exposes tools other than the expected two (or tools that shadow local ones).

In [2]:
async def discover():
    async with mc.connect("stats") as c:
        tools = (await c.session.list_tools()).tools
        resources = (await c.session.list_resources()).resources
        prompts = (await c.session.list_prompts()).prompts
        return c.protocol_version, tools, resources, prompts


version, tools, resources, prompts = await discover()
print("negotiated protocol version:", version)
for t in tools:
    print(f"\nTOOL {t.name}\n  {t.description[:160]}...")
    for k, v in t.inputSchema["properties"].items():
        enum = v.get("enum")
        print(f"    {k:20} {v.get('description', '')[:70]}" + (f"  enum={enum[:4]}..." if enum else ""))
print("\nRESOURCES:", [str(r.uri) for r in resources])
print("PROMPTS:", [p.name for p in prompts])

negotiated protocol version: 2025-11-25

TOOL query_statistic
  Fetch published statistics from the MOM data in the store. Returns the rows plus unit, seasonal-adjustment basis, residency, source and release date, and a quer...
    indicator            Which statistic.  enum=['unemployment_rate', 'unemployed_persons', 'long_term_unemployment_rate', 'long_term_unemployed_persons']...
    period_from          First period, 'YYYY' or 'YYYY-Qn'.
    period_to            Last period, same form as period_from.
    residency            overall, resident or citizen (not every indicator has one).  enum=['overall', 'resident', 'citizen']...
    age_group            Age band; only for resident unemployment by age.  enum=['15-29', '30-39', '40-49', '50 and over']...
    industry             Industry name as in stats://schema (retrenchments and vacancies only).
    seasonal_adjustment  sa, nsa, or unspecified (the source does not say).  enum=['sa', 'nsa', 'unspecified']...

TOOL compute_change
  Co

## 2. Use a tool, then compute on its result
`query_statistic` returns the rows plus the unit, residency, seasonal-adjustment basis, source and release date, and a `query_id`. `compute_change` takes that id, so the arithmetic is done by code and shows its formula.

In [3]:
async def demo():
    async with mc.connect("stats") as c:
        q, wrapped = await c.call(
            "query_statistic",
            {
                "indicator": "unemployment_rate",
                "period_from": "2025-Q1",
                "period_to": "2026-Q2",
                "residency": "overall",
                "seasonal_adjustment": "sa",
            },
        )
        ch, _ = await c.call("compute_change", {"method": "pp", "query_id": q.query_id})
        return q, ch, wrapped


q, ch, wrapped = await demo()
print(f"{q.unit} | residency={q.residency} | basis={q.seasonal_adjustment} | {q.source_id} | released {q.release_date}")
for r in q.rows:
    print(f"  {r.period}: {r.value}")
print("\nchange:", ch.result, ch.unit, "| formula:", ch.formula, "| inputs:", ch.inputs)
print("\nwhat the agent's prompt would contain:\n", wrapped[:420])

% | residency=overall | basis=sa | unemp_rate_overall_q | released 2026-09-29
  2025-Q1: 2.0
  2025-Q2: 2.0
  2025-Q3: 2.0
  2025-Q4: 2.0
  2026-Q1: 2.0
  2026-Q2: 1.9

change: -0.1 pp | formula: last - first (percentage points) | inputs: {'first': 2.0, 'last': 1.9, 'first_period': '2025-Q1', 'last_period': '2026-Q2'}

what the agent's prompt would contain:
 <untrusted_tool_output source="stats.query_statistic">
{
 "query_id": "q-d21812c14dbe",
 "rows": [
  {
   "period": "2025-Q1",
   "value": 2.0,
   "residency": "overall",
   "age_group": null,
   "industry": null,
   "industry_level": null,
   "seasonal_adjustment": "sa",
   "source_id": "unemp_rate_overall_q",
   "release_type": "latest_revision",
   "release_date": "2026-09-29"
  },
  {
   "period": "2025-Q2",
   "


## 3. The server refuses to guess
Leaving out the residency returns every residency and basis, with a warning, instead of picking one.

In [4]:
async def ambiguous():
    async with mc.connect("stats") as c:
        q, _ = await c.call(
            "query_statistic", {"indicator": "unemployment_rate", "period_from": "2025-Q2", "period_to": "2025-Q2"}
        )
        return q


q = await ambiguous()
print("residency:", q.residency, "| basis:", q.seasonal_adjustment, "| rows:", len(q.rows))
for w in q.warnings:
    print("WARNING:", w)

residency: mixed | basis: mixed | rows: 6


## 4. Resources
`stats://schema` says what exists; `stats://sources` carries provenance and licence.

In [5]:
async def resources_demo():
    async with mc.connect("stats") as c:
        schema, _ = await c.read("stats://schema")
        sources, _ = await c.read("stats://sources")
        return schema, sources


schema, sources = await resources_demo()
print("scope window:", schema.scope_window)
for i in schema.indicators[:4]:
    print(" ", i)
print("\n", len(sources.sources), "sources; licence of the first:", sources.sources[0].licence)

scope window: ['2021-Q3', '2026-Q2']
  {'indicator': 'job_vacancies', 'unit': 'persons', 'period_type': 'quarter', 'first_period': '2021-Q3', 'last_period': '2026-Q2', 'rows': 860}
  {'indicator': 'job_vacancies', 'unit': 'persons', 'period_type': 'year', 'first_period': '2022', 'last_period': '2025', 'rows': 4}
  {'indicator': 'long_term_unemployed_persons', 'unit': 'persons', 'period_type': 'year', 'first_period': '2022', 'last_period': '2025', 'rows': 4}
  {'indicator': 'long_term_unemployment_rate', 'unit': '%', 'period_type': 'year', 'first_period': '2022', 'last_period': '2025', 'rows': 4}

 21 sources; licence of the first: Singapore Open Data Licence v1.0 (https://data.gov.sg/open-data-licence)
